In [1]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel

/home/ubuntu/project/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ubuntu/project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/project/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


🦥 Unsloth Zoo will now patch everything to make training faster!


## Задание 1
В коде ниже реализованы матрицы для LoRA. Напишите код метода `.forward()`, в котором применяются основные веса и добавляются матрицы LoRA.

Выполните задание локально, в своём окружении, а затем сверьтесь с авторским решением.

In [2]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, alpha=16, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / max(1, r)
        # базовый линейный слой замораживаем
        self.base = nn.Linear(in_features, out_features, bias=bias)
        for p in self.base.parameters():
            p.requires_grad_(False)
        # матрицы A и B
        self.A = nn.Linear(in_features, r, bias=False)
        self.B = nn.Linear(r, out_features, bias=False)


    def forward(self, x):
        base_out = self.base(x)
        update = self.B(self.A(x)) * self.scaling
        return base_out + update

        
lora = LoRALinear(128, 128, 4, 16)
x = torch.randn(2, 5, 128)
print(lora(x).size())

torch.Size([2, 5, 128])


## Задание 2
Вы уже реализовали механизм `LoRA`, а теперь реализуйте метод `P-tuning`.

Внимательно прочитайте поясняющие комментарии в коде и выполните задание локально, в своём окружении, а затем сверьтесь с авторским решением. 
- давайте в задании 2 чуть подробнее расскажем, что должно получиться в каждой переменной и что должно возвращаться с форварда

In [3]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM

class SoftPrompt(nn.Module):
    def __init__(self, k=20, d=768):
        super().__init__()
        self.k = k
        self.emb = nn.Parameter(torch.randn(k, d)) # k обучаемых виртуальных токенов

    def forward(self, input_ids, model_embed):
        # input_ids: (B, L) - входная последовательность 
        # model_embed - метод подсчета оригинальных эмбеддингов в модели

        batch_size, seq_len = input_ids.shape

        # 1) посчитайте оригинальные эмбеддинги для входной последовательности
        tok_emb = model_embed(input_ids) # ваш код тут # (B, L, d) 
        # 2) посчитайте добавочные обучаемые (приведите их к нужной размерности)
        soft = self.emb.unsqueeze(0).expand(batch_size, -1, -1) # ваш код тут # (B, k, d)  
        # 3) сконкатенируйте и верните результат 
        return torch.concat([tok_emb, soft], dim=1)  # (B, k+L, d)

model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
p_tuning = SoftPrompt(k=20, d=model.config.hidden_size)
input_ids = torch.tensor([[0, 1, 2, 3], [4, 5, 6, 7]])
output = p_tuning(input_ids, model.get_input_embeddings())
print(output.size())

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3719.71it/s]


torch.Size([2, 24, 896])


## Инструменты для обучения unsloth и peft

Многие методы для трансформеров уже реализованы, например, в библиотеке PEFT, применимой для самых разных адаптеров. Чтобы уменьшить объём кода для эффективного обучения, создали библиотеку Unsloth {target="_blank"}— надстройка над transformers и PEFT. Она ускоряет обучение за счёт эффективных операций в трансформерах. 

Разберём настройку модели на примере:
1. Создадим модель и адаптер:

In [4]:
# from unsloth import FastLanguageModel
import torch

model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    # load_in_8bit=True, # будем использовать int8 при обучении
    # load_in_4bit=False,
    load_in_8bit=False,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(model,
                                        r=8, # ранг
                                        lora_alpha=16, # вес добавления адаптера
                                        # слои, к которым применяем 
                                        target_modules=["q_proj", "k_proj", "v_proj"],
                                        use_gradient_checkpointing=False,
                                    )

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.581 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 819.68it/s]


unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.5.2 patched 24 layers with 24 QKV layers, 0 O layers and 0 MLP layers.


Мы получили обычную модель с точки зрения методов работы с ней.

В прошлом уроке мы обучали модель с помощью DPO из trl, а в этом переиспользуем опыт, но применим метод SFT. 

## Задание 3
В этом задании вы будете работать с примерами запросов из прошлого урока и ответами к ним. 

Трансформируйте эти данные в колонку `'messages'`, дописав функцию `examples_to_messages()`, как в прошлом уроке. Затем изучите документацию на `SFTTrainer`, в том числе его гиперпараметры, и запустите обучение. Обратите внимание на параметры `learning_rate`, `per_device_train_batch_size`,`max_length`, `num_train_epochs`,  `report_to`,  `logging_steps`, `save_strategy`,`dataset_text_field`, `gradient_accumulation_steps` для `SFTConfig` .

В этом задании вам не потребуется сохранять логи и чекпойнты, поэтому параметры, нацеленные на это, нам пока не интересны.


In [6]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
examples = [
    {
        "prompt": "Объясни, почему небо голубое.",
        "chosen": "Потому что молекулы воздуха рассеивают короткие волны света сильнее длинных, поэтому мы видим преимущественно голубую часть спектра.",
    },
    {
        "prompt": "Дай безопасный совет по хранению паролей.",
        "chosen": "Используйте менеджер паролей и включите двухфакторную аутентификацию; не повторяйте один и тот же пароль на разных сайтах.",
    },
]


def examples_to_messages(examples):
    # функция, которая формирует сообщения в колонку messages
    data = {"messages": []}
    for example in examples:
        data["messages"].append([
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["chosen"]},
        ])
    return Dataset.from_dict(data)


ds = examples_to_messages(examples)
# unsloth требует, чтобы в датасете уже были тексты со спецтокенами
ds = ds.map(lambda x: {'text': tokenizer.apply_chat_template(x['messages'], tokenize=False)})

config = SFTConfig(
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    max_length=512,
    num_train_epochs=10,
    report_to='none',
    logging_steps=1,
    save_strategy='no',
    dataset_text_field = "text",
    gradient_accumulation_steps=1,
)

trainer = SFTTrainer(
    model,
    args=config,
    train_dataset=ds,
    processing_class=tokenizer,
)

trainer.train()

Map: 100%|██████████| 2/2 [00:00<00:00, 416.39 examples/s]
num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
[datasets.arrow_dataset|WARNING]num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 2/2 [00:05<00:00,  2.52s/ examples]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 10 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 737,280 of 494,770,048 (0.15% trained)


Step,Training Loss
1,2.913841
2,3.194091
3,3.194091
4,2.913841
5,3.154565
6,2.807448
7,3.013973
8,2.697121
9,2.648561
10,2.855908


TrainOutput(global_step=20, training_loss=2.7434194803237917, metrics={'train_runtime': 7.5272, 'train_samples_per_second': 2.657, 'train_steps_per_second': 2.657, 'total_flos': 3701117245440.0, 'train_loss': 2.7434194803237917, 'epoch': 10.0})